In [7]:
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from flyvis.task.vel_decoder import VelocityTemporalDecoder, second_diff_loss
from flyvis.network import Network
import flyvis
import matplotlib.pyplot as plt

In [8]:
# ----------------------------
# 1. Network + decoder setup
# ----------------------------
network_view = flyvis.NetworkView(flyvis.results_dir / "flow/0000/000")
network = network_view.init_network()
# network.train()
# for p in network.parameters():
#     p.requires_grad_(True)   # freeze pretrained network

decoder = VelocityTemporalDecoder(network.connectome)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

network.to(device)
decoder.to(device)

[2025-12-04 12:21:15] network_view:122 Initialized network view at /home/guardomayas/flyvis_data/results/flow/0000/000
[2025-12-04 12:21:19] network:222 Initialized network with NumberOfParams(free=734, fixed=2959) parameters.
[2025-12-04 12:21:19] chkpt_utils:36 Recovered network state.
[2025-12-04 12:21:21] vel_decoder:68 Initialized VelocityTemporalDecoder with NumberOfParams(free=193, fixed=0) parameters, kernel_size=21


Using device: cpu


VelocityTemporalDecoder(
  (temp_conv): Conv1d(8, 8, kernel_size=(21,), stride=(1,), padding=(10,), groups=8, bias=False)
  (norm): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear): Linear(in_features=8, out_features=1, bias=True)
)

In [9]:
save_path = Path("results/pre_trained_network_decoder.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(save_path, map_location=device)

network.load_state_dict(checkpoint["network_state_dict"])
decoder.load_state_dict(checkpoint["decoder_state_dict"])

train_errors = checkpoint["train_errors"]
val_errors   = checkpoint["val_errors"]



## Moving Edge responses

In [11]:
from flyvis.datasets.moving_bar import MovingEdge
import numpy as np
# initialize dataset
# make the dataset
dataset = MovingEdge(
    offsets=[-10, 11],  # offset of bar from center in 1 * radians(2.25) led size
    intensities=[0, 1],  # intensity of bar
    speeds=[19],  # speed of bar in 1 * radians(5.8) / s
    height=80,  # height of moving bar in 1 * radians(2.25) led size
    post_pad_mode="continue",  # for post-stimulus period, continue with the last frame of the stimulus
    t_pre=1.0,  # duration of pre-stimulus period
    t_post=1.0,  # duration of post-stimulus period
    dt=1 / 200,  # temporal resolution of rendered video
    angles=list(np.arange(0, 360, 30)),  # motion direction (orthogonal to edge)
)

building stimuli:   0%|          | 0/24 [00:00<?, ?it/s]